# I-01: the complete experiment with Run All

Start a **fresh online T4 ?2 session**, attach only the matching **i01-source.zip**,
enable Internet and press **Run All**. No old checkpoint, forms, pilot approval
buttons or batch acknowledgements are required. Each ordinary cell performs its
named finite stage and can be run individually.

Setup ? fixed execution plan ? readiness/direct qualification/headroom ? all
three pilots/objective checks ? measured main admission ? all ten fixed main
batches (twenty worlds) ? full reports/figures ? verified archive ? worker shutdown.
Models stay resident throughout the research stages. Thinking is not selected.

The fresh study starts at **0/500 physical attempts**; direct-only first attempts
are at most **460**, with at most **28 contingency** attempts. Wrong mathematics
is measured, never an admission threshold. Strict protocol/evidence/runtime and
resource gates still stop safely. Every pilot, batch and settled error exports a
cumulative checkpoint. No existing outcome is regenerated on rerun.

The default is a **700-minute planning window** within your reported approximately
12-hour fresh session, leaving a 20-minute margin. This is an estimate, not a
verified provider quota/deadline. Reduce it below if time has already elapsed or
Kaggle shows less remaining. Setup/downloads consume the same fixed window.
Human communication/mechanism review remains retrospective and explicitly pending.


## 1. Fresh-session configuration

Defaults assume a new approximately twelve-hour allocation. Check actual GPU allowance; lower the minutes for any time already elapsed. The evidence note describes an estimate truthfully. Do not copy a previous session deadline or extend the window by rerunning.


In [1]:
RUN_DIR = "/kaggle/working/i01-run-all-study"
SESSION_REMAINING_MINUTES = 700  # Conservative fresh-session estimate; reduce to actual time left.
SESSION_ALLOWANCE_EVIDENCE = "User reports an approximately 12-hour session; 700-minute fresh-session planning bound, provider quota/deadline not independently verified."


## 2. Verify source and inspect the workspace

Only the matching whole-study source release is accepted. Inventory only; no model loading or inference.


In [2]:
from pathlib import Path
import hashlib, json, shutil, sys, tempfile, zipfile

# Select one exact small source attachment. ZIP and Kaggle-expanded roots work.
SOURCE_BUNDLE = None  # Set only if more than one candidate is attached.
source_candidates = list(Path('/kaggle/input').rglob('i01-source.zip'))
source_candidates += [p.parent for p in Path('/kaggle/input').rglob('source-release.json')]
# Expanded old checkpoints carry their own historical source. They are evidence,
# not candidates for this repaired notebook's active source attachment.
def repaired_source_manifest(path):
    try:
        if path.is_dir():
            manifest = json.loads((path/'source-release.json').read_text())
        else:
            with zipfile.ZipFile(path) as archive:
                manifest = json.loads(archive.read('source-release.json'))
        return manifest if manifest.get('execution_policy') == 'i01-whole-study-run-all-2026-10-15' and all('recovery/i01/'+name+'.py' in manifest['files'] for name in ('policy','continuation')) else None
    except (OSError, ValueError, KeyError, zipfile.BadZipFile):
        return None
unique_sources = {}
for candidate in source_candidates:
    manifest = repaired_source_manifest(candidate)
    if manifest:
        identity = (manifest.get('git_head'), json.dumps(manifest['files'], sort_keys=True))
        unique_sources.setdefault(identity, candidate)  # Same ZIP/expanded release is one choice.
source_candidates = list(unique_sources.values())
if SOURCE_BUNDLE is None:
    if len(source_candidates) != 1:
        raise RuntimeError(f'Select SOURCE_BUNDLE explicitly; candidates: {source_candidates}')
    SOURCE_BUNDLE = source_candidates[0]
selected_source = Path(SOURCE_BUNDLE)
with tempfile.TemporaryDirectory(prefix='i01-source-', dir='/kaggle/working') as staging_name:
    staging = Path(staging_name)
    if selected_source.is_file():
        with zipfile.ZipFile(selected_source) as source_zip:
            infos = source_zip.infolist()
            if sum(p.file_size for p in infos) > 64 * 2**20 or len({p.filename for p in infos}) != len(infos):
                raise RuntimeError('Source attachment size or duplicate member violation')
            for item in infos:
                p = Path(item.filename)
                if p.is_absolute() or '..' in p.parts or '\\' in item.filename or ':' in item.filename or ((item.external_attr >> 16) & 0o170000) == 0o120000:
                    raise RuntimeError('Unsafe source archive member')
            if source_zip.testzip() is not None:
                raise RuntimeError('Source ZIP integrity failed')
            source_zip.extractall(staging)
    elif selected_source.is_dir():
        if any(p.is_symlink() for p in selected_source.rglob('*')):
            raise RuntimeError('Select an ordinary verified expanded source directory')
        shutil.copytree(selected_source, staging, dirs_exist_ok=True)
    else:
        raise RuntimeError('Missing exact selected source attachment')
    release = json.loads((staging/'source-release.json').read_text())
    if release['format'] != 'i01-online-source-v1':
        raise RuntimeError('Wrong source release format')
    if release.get('execution_policy') != 'i01-whole-study-run-all-2026-10-15' or not all('recovery/i01/'+name+'.py' in release['files'] for name in ('policy','continuation')):
        raise RuntimeError('This notebook requires the whole-study Run All source packet; preserve old checkpoints and replace the active source attachment.')
    head = release.get('git_head', '')
    if len(head) != 40 or any(c not in '0123456789abcdef' for c in head):
        raise RuntimeError('Invalid source release commit')
    source_root = Path('/kaggle/working') / ('i01-source-' + head[:12])
    for relative, wanted in release['files'].items():
        path = staging/relative
        if not path.is_file() or path.stat().st_size != wanted['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != wanted['sha256']:
            raise RuntimeError(f'Source identity failed: {relative}')
        if source_root.exists() and (not (source_root/relative).is_file() or hashlib.sha256((source_root/relative).read_bytes()).hexdigest() != wanted['sha256']):
            raise RuntimeError('Existing source differs; select a fresh session/source directory')
    if not source_root.exists():
        shutil.copytree(staging, source_root)
loaded_source = sys.modules.get('recovery')
if loaded_source is not None and not Path(loaded_source.__file__).resolve().is_relative_to(source_root.resolve()):
    raise RuntimeError('Restart the notebook kernel before using this source release; saved study files are preserved.')
sys.path.insert(0, str(source_root))
from recovery.i01.notebook import Study
if 'study' not in globals():
    study = Study(source_root, run=RUN_DIR)
elif study.source.resolve()!=source_root.resolve() or study.run_dir.resolve()!=Path(RUN_DIR).resolve():
    raise RuntimeError('Finish the current study and restart the kernel before changing source or RUN_DIR; existing study files are preserved.')
study.bootstrap_status()


Workspace,Observed value
Python,"3.13.15 (main, Aug 6 2026, 11:06:22) [GCC 13.3.0]"
System,Linux
Source,/kaggle/working/i01-source-39ad71bf6b00
Study files,/kaggle/working/i01-run-all-study


Device,Model,UUID,Memory (MiB)
0,Tesla T4,GPU-205cab9c-5963-e06e-8788-cd1bad6e5c13,15360
1,Tesla T4,GPU-c7b2971b-40ee-7e1e-0400-43d5b05bffe4,15360


Storage,Path,Free GiB
scratch,/kaggle/temp/i01,1069.9
working,/kaggle/working,19.5


Session window: unset; preparation is available, model loading requires actual facts in its dedicated cell.
Next: run Install the pinned runtime. Host acceptance still occurs before installation.


## 3. Record the fixed execution window

Record BEFORE installation and downloads. Reruns retain this deadline. No provider assignment, quota or release is inferred from the configured planning window.


In [3]:
study.configure_session(remaining_minutes=SESSION_REMAINING_MINUTES,
    evidence=SESSION_ALLOWANCE_EVIDENCE);


Measurement,Recorded value
Deadline,2026-10-16T02:31:59.412575+00:00
Evidence,"User reports an approximately 12-hour session; 700-minute fresh-session planning bound, provider quota/deadline not independently verified."
Apply after initialization,Yes


## 4. Install the pinned runtime

Verified pip bootstrap without ensurepip, unchanged pinned dependencies, isolated/headless controller, visible logs/progress. No host startup hooks or unpinned upgrades.


In [4]:
study.install_runtime();


Setup: venv
Setup: download verified pip bootstrap wheel
Setup: bootstrap-pip
Setup: pip-isolation
Setup: dependencies
dependencies: 15s elapsed;      ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 116.3 MB/s  0:00:05
dependencies: 30s elapsed;      ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 94.7 MB/s  0:00:02
dependencies: 45s elapsed;   Downloading nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl (287.2 MB)
dependencies: 60s elapsed;   Downloading tenacity-9.2.1-py3-none-any.whl (32 kB)
dependencies: 75s elapsed;   Downloading triton-3.6.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.3 MB)
dependencies: 90s elapsed; py, anyio, rich, requests-toolbelt, pydantic, nvidia-cusolver-cu12, matplotlib, httpx2, httpx, typer, langsmith, torch, langchain-core, huggingface-hub, tokenizers, langgraph-sdk, langgraph-checkpoint, bitsandbytes, accelerate, transformers, langgraph-prebuilt, langgraph-checkpoint-sqlite, langgraph
dependencies: 105

Measurement,Recorded value
Interpreter,/kaggle/temp/i01/i01-venv-39ad71bf6b00/bin/python
Source,/kaggle/working/i01-source-39ad71bf6b00
Scratch,/kaggle/temp/i01
Working,/kaggle/working
Hf cache,/kaggle/temp/i01/hf-cache
Setup seconds,185.614963617
Model attempts,0


## 5. Start the independent study

New study at zero, no old checkpoint/result import. Repeating this cell reuses the same destination and preserves spending; an old shared campaign cannot be converted.


In [5]:
study.start_new();


Measurement,Recorded value
Deadline,2026-10-16T02:31:59.412575+00:00
Evidence,"User reports an approximately 12-hour session; 700-minute fresh-session planning bound, provider quota/deadline not independently verified."
Apply after initialization,Yes


session: recorded in the stage receipt
Checkpoint action/window: 1,433,508 bytes; integrity passed
window: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip
Checkpoint study-open: 1,433,534 bytes; integrity passed
snapshot: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


Measurement,Recorded value
Attempt scope,independent-study
Attempts,0
Attempts unknown generated tokens,0
Attempts unknown ready seconds,0
Campaign attempts,0
Campaign attempts unknown ready seconds,0
Campaign generated tokens observed,0
Campaign ready seconds observed,0
Clear minus retain,Unknown / not recorded
Complete case descriptive mean,Unknown / not recorded


Main coverage,Value
Known paired outcomes,0 / 20
Campaign physical attempts,0 / 500 (0 here)
Clear minus retain,Unknown / not recorded
Unknown-outcome bounds,"[-1.0, 1.0]"


Logical slot,Resident,Load receipt
0,No,Not recorded
1,No,Not recorded


Budget,Value
attempt scope,independent-study
attempts,0
campaign attempts,0
contingency,0
in flight,0
local contingency,0
lost spending,"{'resolved': True, 'upper': 0}"
missing prior studies,[]
prior attempts,0
prior spending pending,Unknown / not recorded


## 6. Check implementation and native tokenizer

Focused scripted math/private-view/lifecycle/accounting/durability/archive checks and native token bounds. Zero model attempts.


In [6]:
study.check_implementation();


Checkpoint action/selfcheck: 1,445,373 bytes; integrity passed
selfcheck: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


Measurement,Recorded value
Environment,scripted temporary DBs; no model loaded
Executed at,2026-10-15T14:55:22.111731+00:00
Model attempts,0
Native tokenizer,Yes
Passed,Yes


Lightweight check,State
native fixture 827/revised 780; duplicates/negative/boundary,Passed
fixed strata/seeds; private views; finality/wait; literal siblings; invalid actions,Passed
raw-before-effects; idempotence; pre-persistence retry; 499/500; backup/ZIP/expanded restore; unknown vs zero,Passed


Native tokenizer: passed


Checkpoint: passed


## 7. Download and verify weights

Pinned model snapshot and actual storage/native file checks. Scratch caches are reused when valid.


In [7]:
study.download_weights();


Setup: weights-command
weights-command: 15s elapsed; Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
weights-command: 75s elapsed; Fetching 12 files:  42%|████▏     | 5/12 [00:51<01:12, 10.34s/it]
weights-command: 90s elapsed; Verifying pinned native bytes 4/12: model.safetensors-00001-of-00004.safetensors (5,276,436,216 bytes)
weights-command: 105s elapsed; Verifying pinned native bytes 4/12: model.safetensors-00001-of-00004.safetensors (5,276,436,216 bytes)
weights-command: 120s elapsed; Verifying pinned native bytes 5/12: model.safetensors-00002-of-00004.safetensors (5,335,161,512 bytes)
weights-command: 135s elapsed; Verifying pinned native bytes 6/12: model.safetensors-00003-of-00004.safetensors (5,368,717,440 bytes)
weights-command: 150s elapsed; Verifying pinned native bytes 6/12: model.safetensors-00003-of-00004.safetensors (5,368,71

Measurement,Recorded value
Model,Qwen/Qwen3.5-9B
Revision,c202236235762e1c871ad0ccb60c8ee5ba337b9a
Snapshot,/kaggle/temp/i01/hf-cache/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a
Native total bytes,19329302519
Weights,Yes
Model attempts,0


## 8. Load both resident T4 workers

Two fixed replicas; concise actual load/device evidence. Loading generates no hidden warmup. Inspection and research stages retain these replicas.


In [8]:
study.load_models();


Working · 60s · Loading weights:  77%|███████▋  | 329/427 [00:47<00:07, 13.17it/s]
Working · 75s · Model load: audit actual tensors, quantization and loading diagnostics
Working · 105s · Loading weights:  93%|█████████▎| 397/427 [00:21<00:01, 17.95it/s]
Checkpoint action/load: 1,506,663 bytes; integrity passed
load: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


Logical slot,CUDA UUID,Load seconds,Total device GiB
0,205cab9c-5963-e06e-8788-cd1bad6e5c13,71.93492409200007,14.56
1,c7b2971b-40ee-7e1e-0400-43d5b05bffe4,33.652908379999985,14.56


Logical slot,Model / revision,Attention,Recurrent fast path
0,Qwen/Qwen3.5-9B / c202236235762e1c871ad0ccb60c8ee5ba337b9a,recovery_t4_sdpa,No
1,Qwen/Qwen3.5-9B / c202236235762e1c871ad0ccb60c8ee5ba337b9a,recovery_t4_sdpa,No


Pinned text-only NF4/FP16 replicas accepted by the reviewed loader. The loading-only generation profile is not the per-call I-01 profile: direct uses 512 output tokens, temperature 0.7/top-p 0.8/top-k 20.
Known upstream documentation/deprecation messages and optional recurrent-kernel fallback are retained in controller-stderr.log. Full load/tensor inventories remain in sessions.csv and the checkpoint; no runtime upgrade is applied.
Checkpoint: passed


## 9. Lock the complete execution plan

Before the first model call, bind the unchanged manifest/settings, objective checks, ten fixed batches and retrospective review policy. Pilots cannot tune this configuration.


In [9]:
study.plan_execution();


Checkpoint action/plan_execution: 1,572,429 bytes; integrity passed
plan_execution: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


Measurement,Recorded value
At,2026-10-15T14:59:59.655273+00:00
External download verified,No
Human review,retrospective_pending
Human understanding verified,No
Main admission,three objective pilot passes and adequate measured resources; no settings tuning
Main batches,10
Mode,direct
Pilot admission,"complete durable phases/controls, original submissions, valid protocol; accuracy diagnostic"
Policy,i01-whole-study-run-all-2026-10-15
Useful communication,Unknown / not recorded


Checkpoint: passed


## 10. Check fresh-state readiness

Six A/B/A calls. Cache/numerical/provider faults or unexplained unequal A/B/A stop; no automatic waiver or extra samples.


In [10]:
study.check_readiness();


hardware/state-0-A1 · GPU slot 0 · attempt 1 · campaign 1/500 (1 here)
Raw saved: hardware/state-0-A1 · 14 tokens · 3.6579696720000356 ready-model seconds
hardware/state-0-B · GPU slot 0 · attempt 2 · campaign 2/500 (2 here)
Raw saved: hardware/state-0-B · 16 tokens · 2.0135626539999976 ready-model seconds
hardware/state-0-A2 · GPU slot 0 · attempt 3 · campaign 3/500 (3 here)
Raw saved: hardware/state-0-A2 · 14 tokens · 1.5698537459999216 ready-model seconds
hardware/state-1-A1 · GPU slot 1 · attempt 1 · campaign 4/500 (4 here)
Raw saved: hardware/state-1-A1 · 14 tokens · 2.559528384000032 ready-model seconds
hardware/state-1-B · GPU slot 1 · attempt 2 · campaign 5/500 (5 here)
Raw saved: hardware/state-1-B · 16 tokens · 2.0032953429999907 ready-model seconds
hardware/state-1-A2 · GPU slot 1 · attempt 3 · campaign 6/500 (6 here)
Raw saved: hardware/state-1-A2 · 14 tokens · 1.5390514629999643 ready-model seconds
Working · 15s · Inference pid=393 device=1 seed=9113091594724103221 phase=d

Measurement,Recorded value
Equality policy,Diagnostic: unexpected differences require explicit state-isolation review before qualification; no extra samples or decoding changes.
Needs review,No
Passed,Yes


Slot,Probe,Raw visible reply,Finite score steps,Fresh cache
0,A1,"{""message"":null,""submit"":3,""wait"":false}",14,Yes
0,B,"{""message"":null,""submit"":100,""wait"":false}",16,Yes
0,A2,"{""message"":null,""submit"":3,""wait"":false}",14,Yes
1,A1,"{""message"":null,""submit"":3,""wait"":false}",14,Yes
1,B,"{""message"":null,""submit"":100,""wait"":false}",16,Yes
1,A2,"{""message"":null,""submit"":3,""wait"":false}",14,Yes


A/B/A equal by slot: [True, True]; Diagnostic: unexpected differences require explicit state-isolation review before qualification; no extra samples or decoding changes.


Checkpoint: passed


## 11. Qualify direct protocol

Twelve fixed original/revised controls. All twelve must be committed and protocol-valid. Wrong scalar answers alone pass protocol admission and remain visible outcomes.


In [11]:
study.qualify_direct();


qualification-00/control-original-direct · GPU slot 0 · attempt 4 · campaign 7/500 (7 here)
Raw saved: qualification-00/control-original-direct · 16 tokens · 2.2565875029999916 ready-model seconds
qualification-00/control-revised-direct · GPU slot 0 · attempt 5 · campaign 8/500 (8 here)
Raw saved: qualification-00/control-revised-direct · 16 tokens · 1.9528101439999546 ready-model seconds
qualification-01/control-original-direct · GPU slot 1 · attempt 4 · campaign 9/500 (9 here)
Raw saved: qualification-01/control-original-direct · 16 tokens · 2.4477364729999636 ready-model seconds
qualification-01/control-revised-direct · GPU slot 1 · attempt 5 · campaign 10/500 (10 here)
Raw saved: qualification-01/control-revised-direct · 16 tokens · 1.9337935099999868 ready-model seconds
qualification-02/control-original-direct · GPU slot 0 · attempt 6 · campaign 11/500 (11 here)
Raw saved: qualification-02/control-original-direct · 17 tokens · 2.403428591000079 ready-model seconds
qualification-02

Measurement,Recorded value
Admitted,12
Attempts unknown effort,0
Committed,12
Correct,10
Generated tokens observed,193
Mode,direct
Passed,Yes
Physical attempts,12
Planned,12
Policy,i01-direct-protocol-2026-10-15


Protocol qualification: 12/12 committed; 0 invalid actions; current eligible: True. Mathematical diagnostic: 10/12.
Original saved gate decision (unchanged): True; rule i01-direct-protocol-2026-10-15


Failed call,Truth,Submission,Error
qualification-03/control-original-direct,951,788,wrong scalar
qualification-03/control-revised-direct,920,788,wrong scalar


Checkpoint: passed


## 12. Inspect qualification

Every planned direct control, truth/submission, validity, correctness, effort and failure-first raw audit. Complete raw evidence is downloadable.


In [12]:
study.inspect_qualification(mode="direct");


Mode,Committed/planned,Correct,Wrong scalar,Invalid,Unexecuted,Current eligible,Historical passed
direct,12/12,10,2,0,0,Yes,Yes


Call,Truth,Submission,Valid,Correct,Error,Tokens,Ready seconds
qualification-00/control-original-direct,950,950,Yes,1,Unknown / not recorded,16,2.2565875029999916
qualification-00/control-revised-direct,921,921,Yes,1,Unknown / not recorded,16,1.9528101439999546
qualification-01/control-original-direct,999,999,Yes,1,Unknown / not recorded,16,2.4477364729999636
qualification-01/control-revised-direct,931,931,Yes,1,Unknown / not recorded,16,1.9337935099999868
qualification-02/control-original-direct,1000,1000,Yes,1,Unknown / not recorded,17,2.403428591000079
qualification-02/control-revised-direct,981,981,Yes,1,Unknown / not recorded,16,1.9104771290000144
qualification-03/control-original-direct,951,788,Yes,0,Unknown / not recorded,16,2.009283848999985
qualification-03/control-revised-direct,920,788,Yes,0,Unknown / not recorded,16,1.9197708560000137
qualification-04/control-original-direct,956,956,Yes,1,Unknown / not recorded,16,2.1762248680000766
qualification-04/control-revised-direct,918,918,Yes,1,Unknown / not recorded,16,1.8883949929999062


Main paired coverage: 0/20 intended worlds. Campaign: 18/500 physical attempts (18 here, 0 prior); contingency including prior charges/gaps: 0/28. Historical 45 remain separate.


Admission resource,Value
Remaining attempt allowance,482
Contingency including bounded lost spending,0
Lost spending evidence,"{'resolved': True, 'upper': 0}"
Remaining session seconds,41472.906263


Showing 3/12 detailed call audits. Complete prompts/replies/tokens remain in conversations.html/.txt/.jsonl. Use study.inspect(..., audit_limit=None) to display all calls.


## 13. Check full headroom

Two full-input/full-output diagnostic probes, one per current load. Require at least 1.5 GiB measured free headroom on each T4.


In [13]:
study.check_headroom();


hardware/envelope-direct-0 · GPU slot 0 · attempt 10 · campaign 19/500 (19 here)
Working · 15s · Inference pid=379 device=0 seed=3554240822569505675 phase=decode elapsed=14.6s input=8192/8192 decode=0/512 peak_reserved_mib=7824.0
Working · 30s · Inference pid=379 device=0 seed=3554240822569505675 phase=decode elapsed=29.7s input=8192/8192 decode=150/512 peak_reserved_mib=7824.0
Working · 45s · Inference pid=379 device=0 seed=3554240822569505675 phase=decode elapsed=44.8s input=8192/8192 decode=300/512 peak_reserved_mib=7824.0
Working · 60s · Inference pid=379 device=0 seed=3554240822569505675 phase=decode elapsed=59.8s input=8192/8192 decode=450/512 peak_reserved_mib=7824.0
Raw saved: hardware/envelope-direct-0 · 512 tokens · 65.99600955200003 ready-model seconds
hardware/envelope-direct-1 · GPU slot 1 · attempt 10 · campaign 20/500 (20 here)
Working · 75s · Inference pid=393 device=1 seed=6895241862355992459 phase=prefill elapsed=0.0s input=0/8192 decode=0/512 peak_reserved_mib=7540.0

Measurement,Recorded value
Mode,direct
Passed,Yes
Reload,No
Session,bca33bff-34fa-4e19-a633-2a90cf36c736


Slot,Generated tokens,Observed free headroom (GiB),Failure
0,512,6.788,None recorded
1,512,6.784,None recorded


Checkpoint: passed


## Run pilot 1

Pilot 1 only: prefix, literal retain/clear siblings and original/revised controls, at most twenty first attempts. Conversation and cumulative checkpoint appear automatically. No outcome-based replacement or extra seed.


In [14]:
study.run_pilot(1);


pilot-00/prefix/agent-0/invocation-0 · GPU slot 0 · attempt 11 · campaign 21/500 (21 here)
Raw saved: pilot-00/prefix/agent-0/invocation-0 · 16 tokens · 2.5251864959999466 ready-model seconds
pilot-00 / prefix / agent 0: {'message': None, 'submit': None, 'wait': True} · continuing
pilot-00/prefix/agent-1/invocation-0 · GPU slot 0 · attempt 12 · campaign 22/500 (22 here)
Raw saved: pilot-00/prefix/agent-1/invocation-0 · 78 tokens · 8.566610566000008 ready-model seconds
pilot-00 / prefix / agent 1: {'message': 'I am Agent 1. My shard contains 21 integers. I will scan for the maximum value in my shard and coordinate with Agent 2 to determine the global maximum. I will submit my candidate for the global maximum if I believe it is the highest value found so far across both shards.', 'submit': 923, 'wait': False} · continuing
pilot-00/prefix/agent-0/invocation-1 · GPU slot 0 · attempt 13 · campaign 23/500 (23 here)
Working · 15s · Inference pid=379 device=0 seed=1023110753385749315 phase=dec

Measurement,Recorded value
Pilot,1
World,pilot-00


Checkpoint: passed


World,Original team,Retain,Clear,Original control,Revised control
pilot-00,Unsuccessful (recorded),Unsuccessful (recorded),Unsuccessful (recorded),Correct,Correct


Main paired coverage: 0/20 intended worlds. Campaign: 32/500 physical attempts (32 here, 0 prior); contingency including prior charges/gaps: 0/28. Historical 45 remain separate.


...

Agent 0 · pilot-00/prefix/agent-0/invocation-0 · waiting


Agent 1 · pilot-00/retain/agent-1/invocation-0 · invalid/failed action; no message delivered: malformed_action


Agent 0 · pilot-00/clear/agent-0/invocation-0 · invalid/failed action; no message delivered: malformed_action


Agent 0 · pilot-00/clear/agent-0/invocation-1 · waiting


Showing 6/12 detailed call audits. Complete prompts/replies/tokens remain in conversations.html/.txt/.jsonl. Use study.inspect(..., audit_limit=None) to display all calls.


## Check pilot 1 automatically

Require complete durable terminal episodes/controls, both original-team submissions, valid protocol, reconciled spending and intact runtime/export evidence. Mathematical errors and message quality do not gate admission. Useful communication remains unreviewed.


In [15]:
study.check_pilot(1);


Stopped check_pilot: ValueError: Automatic pilot check failed: valid_protocol
Checkpoint action/check_pilot: 1,757,166 bytes; integrity passed
check_pilot: ValueError: Automatic pilot check failed: valid_protocol · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


Checkpoint: passed
Checkpoint operator-error-check_pilot: 1,757,535 bytes; integrity passed
snapshot: complete · checkpoint /kaggle/working/i01-run-all-study/exports/latest.zip


ActionStopped: ValueError: Automatic pilot check failed: valid_protocol

## Run pilot 2

Pilot 2 only: prefix, literal retain/clear siblings and original/revised controls, at most twenty first attempts. Conversation and cumulative checkpoint appear automatically. No outcome-based replacement or extra seed.


In [ ]:
study.run_pilot(2);


## Check pilot 2 automatically

Require complete durable terminal episodes/controls, both original-team submissions, valid protocol, reconciled spending and intact runtime/export evidence. Mathematical errors and message quality do not gate admission. Useful communication remains unreviewed.


In [ ]:
study.check_pilot(2);


## Run pilot 3

Pilot 3 only: prefix, literal retain/clear siblings and original/revised controls, at most twenty first attempts. Conversation and cumulative checkpoint appear automatically. No outcome-based replacement or extra seed.


In [ ]:
study.run_pilot(3);


## Check pilot 3 automatically

Require complete durable terminal episodes/controls, both original-team submissions, valid protocol, reconciled spending and intact runtime/export evidence. Mathematical errors and message quality do not gate admission. Useful communication remains unreviewed.


In [ ]:
study.check_pilot(3);


## Admit the unchanged main experiment

All three objective pilot checks must pass. Record measured attempt/time feasibility under the already locked configuration. No human approval is invented and no pilot-driven tuning occurs. An inadequate forecast stops with a partial checkpoint.


In [ ]:
study.admit_main();


## Main batch 1/10 ? worlds 00 and 01

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(1);


## Main batch 2/10 ? worlds 02 and 03

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(2);


## Main batch 3/10 ? worlds 04 and 05

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(3);


## Main batch 4/10 ? worlds 06 and 07

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(4);


## Main batch 5/10 ? worlds 08 and 09

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(5);


## Main batch 6/10 ? worlds 10 and 11

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(6);


## Main batch 7/10 ? worlds 12 and 13

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(7);


## Main batch 8/10 ? worlds 14 and 15

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(8);


## Main batch 9/10 ? worlds 16 and 17

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(9);


## Main batch 10/10 ? worlds 18 and 19

Execute these two predetermined worlds, one per logical slot, at most 38 first attempts. Commit all terminal outcomes, show conversations and export before the next cell. Wrong/invalid recorded main replies remain measured failures. Rerunning this numbered cell never advances another batch or resamples.


In [ ]:
study.run_main_batch(10);


## Build complete or honest partial reports

All twenty intended worlds, paired outcomes/uncertainty, costs, protocol/qualification/pilot checks, full conversations/raw calls, failure analysis, limitations, tables and eight SVG/PNG figure pairs. Retrospective human review is visibly pending.


In [ ]:
study.build_report();


## Verify the portable checkpoint and download results

Quiescent cumulative ZIP and exact read-only fresh-directory reconstruction. Download latest.zip and the executed notebook. Checkpoints happen automatically even if you ignore their links during execution.


In [ ]:
study.save_checkpoint(verify=True);


## Close model workers and controller

Close after final export; the closing checkpoint retains actual process evidence. Stop Kaggle through its UI to release its allocation. Process closure and a link do not prove provider deallocation or an external download.


In [ ]:
study.finish_session();


## Results, errors and later restoration

The cumulative checkpoint is **/kaggle/working/i01-run-all-study/exports/latest.zip**;
reports, all conversations/raw outcomes, CSV tables and figures are under
**/kaggle/working/i01-run-all-study/reports/**. Download them and the executed
notebook outside the session. Save existing output without another inference run.

A real prerequisite failure stops further admission and keeps the last valid
archive plus receipts. A provider hard shutdown may occur before a new export;
no missing outcome or unbounded spending is silently assumed zero. Checkpoints
are already present; a convenient restore workflow can be added later without
having to reconstruct unsaved hours of work.

For inspection with intact files (copy only the needed commands into a cell):

    study.show_status();
    study.inspect(audit_limit=None);  # Every raw call, no display limit.
    study.links(all_files=True);
    study.build_report();  # Works read-only after controller closure.

Before closure, interrupt requests pause; wait for admitted calls to settle:

    study.pause();
    study.save_checkpoint(verify=True);
    study.finish_session();

For a later session, use this checkpoint's exact source and the existing explicit
restore/reopen commands in **docs/I01_KAGGLE.md**. Supply actual prior-exit/tail
and current-time facts. New loaded replicas require two charged reload probes;
do not repeat completed controls, pilot outcomes or earlier numbered batches.
Do not rerun the complete notebook to bypass a failed gate or reset its budget.

After execution, inspect all pilot/main conversations and add genuine human
annotations using the frozen rubric and exact call references. Automatic checks
establish evidence/protocol/resource eligibility, not meaningful communication,
human understanding, download completion or a favorable scientific effect.
